# Wie ein Test funktioniert

**Notebook 3 von 5.** Ein Bestimmungsbuch behauptet, dass Zügelpinguine (Chinstrap) im Schnitt
**3.700 g** wiegen. Unsere 68 Zügelpinguine wiegen im Schnitt **3.733 g**. Am Ende von
`02_sampling_and_confidence_intervals.ipynb` hast du diese Behauptung mit einem
Konfidenzintervall beurteilt: 3.700 g lag darin, war also plausibel.

Dieses Notebook stellt dieselbe Frage andersherum, und genau so stellt sie jeder
Hypothesentest: **Wenn das Bestimmungsbuch recht hätte, wie überraschend wären dann unsere 68
Pinguine?** Du beantwortest sie von Hand, Schritt für Schritt, nur mit dem, was die letzten
beiden Notebooks aufgebaut haben: einem Standardfehler aus Notebook 02 und einer Fläche unter
einer Kurve aus Notebook 01. Danach bekommst du dieselbe Antwort aus einer einzigen Zeile scipy,
und ab dann ist diese Zeile alles, was du brauchst.

Am Ende solltest du Folgendes können:

- für eine Frage an die Daten eine Nullhypothese und eine Alternativhypothese formulieren.
- eine t-Statistik von Hand berechnen und sagen, was sie misst.
- einen p-Wert als Fläche unter der t-Verteilung berechnen und sagen, was er bedeutet.
- entscheiden, ob die Nullhypothese verworfen wird, und die Schlussfolgerung korrekt formulieren.
- denselben Test mit `scipy.stats.ttest_1samp` durchführen und ihn einem Konfidenzintervall
  zuordnen.
- vor dem Blick auf die Daten zwischen einem einseitigen und einem zweiseitigen Test wählen.

In [ ]:
# Datenanalyse
import numpy as np
import pandas as pd

# Visualisierung
import matplotlib.pyplot as plt
import seaborn as sns

# Statistik
from scipy import stats

sns.set_style("whitegrid")

In [ ]:
df_penguins = pd.read_csv("data/penguins.csv")

chinstrap = df_penguins[df_penguins["species"] == "Chinstrap"]
mass = chinstrap["body_mass_g"]

## 1. Die Logik: Nimm die langweilige Erklärung an

Unsere Stichprobe ist 33 g schwerer, als das Bestimmungsbuch sagt. Dafür gibt es zwei
Erklärungen:

- **Die langweilige.** Das Bestimmungsbuch hat recht, und 33 g sind nur das Schwanken eines
  Mittelwerts aus 68 Pinguinen, das Schwanken, das Notebook 02 gemessen hat.
- **Die interessante.** Das Bestimmungsbuch liegt falsch, und Zügelpinguine wiegen im Schnitt
  tatsächlich etwas anderes als 3.700 g.

Ein Hypothesentest nimmt die langweilige Erklärung, **geht davon aus, dass sie stimmt**, und
fragt, wie überraschend die Daten unter dieser Annahme wären. Wären die Daten sehr
überraschend, steckt die langweilige Erklärung in Schwierigkeiten. Wenn nicht, bleibt sie
bestehen.

Das funktioniert wie vor Gericht. Für die angeklagte Person gilt die Unschuldsvermutung, und die
Frage lautet nicht *„Hat sie es getan?“*, sondern *„Sind die Beweise stark genug, um die Vermutung
aufzugeben?“*

Die beiden Erklärungen bekommen formale Namen:

- die **Nullhypothese**, $H_0$: die langweilige, *es passiert nichts*. Hier $\mu = 3{.}700$ g.
- die **Alternativhypothese**, $H_1$: das, worauf wir stattdessen schließen würden. Hier
  $\mu \neq 3{.}700$ g.

Beide sind Aussagen über die **Grundgesamtheit**, alle Zügelpinguine, nie über unsere
Stichprobe. Dass die Stichprobe im Schnitt 3.733 g wiegt, wissen wir schon; die Frage ist, was
das über $\mu$ aussagt.

Eine Entscheidung muss fallen, bevor du irgendein Ergebnis ansiehst: Wie überraschend ist
überraschend genug? Diese Schwelle ist das **Signifikanzniveau** $\alpha$, und 0,05 ist die
übliche Wahl. Behalte es im Kopf; Abschnitt 4 erklärt genau, was es bedeutet.

In [ ]:
# die Behauptung des Bestimmungsbuchs: der Wert von mu unter der Nullhypothese
mu_0 = 3700

# wie überraschend überraschend genug ist, festgelegt vor dem Test
alpha = 0.05

## 2. Die Teststatistik: Wie viele Standardfehler entfernt?

Die Stichprobe liegt 33 g von der Behauptung entfernt. Ob 33 g viel sind, hängt davon ab, wie
stark ein Mittelwert aus 68 Pinguinen schwankt, und das ist der Standardfehler aus Notebook 02.
Also teilst du das eine durch das andere:

$$
t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}} = \frac{\text{Stichprobenmittelwert} - \text{behaupteter Mittelwert}}{\text{Standardfehler}}
$$

Das ist die **t-Statistik**. Sie ist der z-Wert aus Notebook 01, angewendet auf einen
Mittelwert: wie viele Standardfehler die Stichprobe von der Behauptung entfernt liegt. Ein Wert
nahe 0 bedeutet, dass die Stichprobe nah an der Behauptung liegt; ein Wert jenseits von etwa 2 in
die eine oder andere Richtung bedeutet, dass sie weit weg liegt.

In [ ]:
n = len(mass)
mean_mass = mass.mean()
se_mass = mass.std() / np.sqrt(n)

# wie viele Standardfehler liegt der Stichprobenmittelwert von der Behauptung entfernt?
t_stat = (mean_mass - mu_0) / se_mass

print(f"Stichprobenmittelwert : {mean_mass:.1f} g")
print(f"Standardfehler        : {se_mass:.1f} g")
print(f"t-Statistik           : {t_stat:.2f}")

Unsere Stichprobe liegt 0,71 Standardfehler über der Behauptung, deutlich weniger als ein
Standardfehler. Auf der Skala, auf die es ankommt, sind 33 g ein kleiner Abstand.

## 3. Wie überraschend ist das? Die t-Verteilung

Wenn die Nullhypothese stimmt und wir die Expedition viele Male wiederholen würden, käme die
t-Statistik jedes Mal ein wenig anders heraus. Wie sie schwanken würde, ist bekannt: Sie folgt
der **t-Verteilung**, die du in den Notebooks 01 und 02 kennengelernt hast, mit
$n - 1 = 67$ Freiheitsgraden.

Das ist die Kurve, mit der wir unser Ergebnis vergleichen. Mit $\alpha = 0{,}05$ werden die
5 Prozent der überraschendsten Ergebnisse auf die beiden Ränder aufgeteilt, 2,5 Prozent auf
jeden, und die Werte, an denen die Ränder beginnen, sind die **kritischen Werte**. Sie sind die
t-Version von 1,96.

In [ ]:
dof = n - 1

# die Werte, die in jedem Rand 2,5 % abschneiden
t_crit = stats.t.ppf(1 - alpha / 2, df=dof)
print(f"kritische Werte: -{t_crit:.3f} und +{t_crit:.3f}")

In [ ]:
x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, df=dof)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, y, color="steelblue", linewidth=2)

# die Ablehnungsbereiche: Ergebnisse, die überraschend genug sind, um H0 aufzugeben
ax.fill_between(x, y, where=(x <= -t_crit), color="crimson", alpha=0.4,
                label=f"Ablehnungsbereiche, je {alpha / 2:.1%}")
ax.fill_between(x, y, where=(x >= t_crit), color="crimson", alpha=0.4)

# wo unsere Stichprobe gelandet ist
ax.axvline(t_stat, color="darkgreen", linestyle="--", linewidth=2,
           label=f"unsere t-Statistik, {t_stat:.2f}")

ax.set_title(f"Die t-Verteilung mit {dof} Freiheitsgraden, wenn H0 stimmt")
ax.set_xlabel("t")
ax.set_ylabel("Dichte")
ax.legend(loc="upper left")
plt.show()

Unsere t-Statistik landet in der Mitte der Kurve, weit weg von den roten Rändern. Wenn das
Bestimmungsbuch recht hat, ist ein Ergebnis wie unseres völlig gewöhnlich.

## 4. Der p-Wert ist eine Fläche

Das Bild liefert das Urteil schon, aber ein Test gibt es als eine einzige Zahl an, den
**p-Wert**:

> **Der p-Wert ist die Wahrscheinlichkeit für ein Ergebnis, das mindestens so extrem ist wie
> unseres, wenn die Nullhypothese stimmt.**

*Mindestens so extrem* heißt mindestens 0,71 Standardfehler von der Behauptung entfernt, in
**beide** Richtungen, denn $H_1$ sagte „anders“, nicht „schwerer“. Der p-Wert ist also die
Fläche unter der Kurve jenseits von $+0{,}71$ plus die Fläche jenseits von $-0{,}71$. Notebook 01
hat gezeigt, wie man die Fläche jenseits eines Werts bekommt, `1 - cdf`, und die beiden Ränder
sind gleich groß:

In [ ]:
# Fläche jenseits von |t| in einem Rand, verdoppelt für beide Ränder
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=dof))
print(f"p-Wert: {p_value:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, y, color="steelblue", linewidth=2)

# der p-Wert: alles, was mindestens so weit von 0 entfernt ist wie unsere t-Statistik
ax.fill_between(x, y, where=(x >= abs(t_stat)), color="darkgreen", alpha=0.35,
                label=f"p-Wert = {p_value:.3f}")
ax.fill_between(x, y, where=(x <= -abs(t_stat)), color="darkgreen", alpha=0.35)
ax.axvline(t_stat, color="darkgreen", linestyle="--", linewidth=2)
ax.axvline(-t_stat, color="darkgreen", linestyle=":", linewidth=2)

ax.set_title("Der p-Wert ist die schattierte Fläche")
ax.set_xlabel("t")
ax.set_ylabel("Dichte")
ax.legend(loc="upper left")
plt.show()

Der p-Wert ist 0,48. In Worten: **Wenn Zügelpinguine wirklich im Schnitt 3.700 g wiegen würden,
fänden 48 Prozent der Expeditionen wie unserer einen Unterschied von mindestens 33 g.** An
unserer Stichprobe ist überhaupt nichts überraschend.

### Die Entscheidung

Die Regel ist kurz: Vergleiche den p-Wert mit $\alpha$.

- $p < \alpha$: Die Daten wären unter $H_0$ überraschend. **Verwirf** $H_0$.
- $p \geq \alpha$: Sie wären es nicht. $H_0$ wird **nicht verworfen**.

Beide Bilder oben zeigen dieselbe Entscheidung. Der p-Wert liegt genau dann unter 0,05, wenn die
t-Statistik in einem roten Rand landet.

In [ ]:
if p_value < alpha:
    print(f"p = {p_value:.3f} < {alpha}: H0 verwerfen")
else:
    print(f"p = {p_value:.3f} >= {alpha}: H0 nicht verwerfen")

Wir **verwerfen** die Nullhypothese **nicht**: Unsere Daten liefern keinen Hinweis darauf, dass
das Bestimmungsbuch falsch liegt.

Achte auf die Formulierung. Wir haben nicht gezeigt, dass Zügelpinguine im Schnitt genau
3.700 g wiegen, und wir dürfen nicht sagen, wir hätten die Nullhypothese *angenommen*. Ein
Freispruch vor Gericht ist kein Urteil *unschuldig*; er sagt nur, dass die Beweise nicht stark
genug waren. Unsere 68 Pinguine passen genauso gut zu einem wahren Mittelwert von 3.690 g oder
3.760 g.

Und jetzt die Bedeutung von $\alpha$. Angenommen, $H_0$ stimmt wirklich. Dann landen trotzdem
5 Prozent der Stichproben durch Pech in einem roten Rand, und der Test verwirft eine wahre
Nullhypothese. **$\alpha$ ist also die Fehlalarmrate, der du zugestimmt hast**, und 0,05 zu
wählen heißt zu akzeptieren, dass einer von zwanzig Tests auf reinem Rauschen falschen Alarm
schlägt. Diese Tatsache kommt in Notebook 04 mit voller Wucht zurück.

| | $H_0$ stimmt | $H_0$ stimmt nicht |
| :--- | :--- | :--- |
| **$H_0$ verwerfen** | Fehlalarm, ein *Fehler 1. Art*, Wahrscheinlichkeit $\alpha$ | richtig |
| **Nicht verwerfen** | richtig | übersehener Effekt, ein *Fehler 2. Art* |

## 5. Derselbe Test in einer Zeile

Alles aus den Abschnitten 2 bis 4 ist genau das, was `scipy.stats.ttest_1samp` macht: ein
**Einstichproben-t-Test**, der den Mittelwert einer Stichprobe mit einem behaupteten Wert
vergleicht.

In [ ]:
result = stats.ttest_1samp(mass, popmean=mu_0)
print(f"t-Statistik: {result.statistic:.3f}")
print(f"p-Wert     : {result.pvalue:.3f}")

Dieselbe t-Statistik und derselbe p-Wert wie von Hand. Ab jetzt verwendest du die eine Zeile,
aber du weißt nun, was sie berechnet, und genau das erlaubt dir, sie zu interpretieren.

### Test und Intervall stimmen immer überein

In Notebook 02 lag 3.700 g im 95-Prozent-Konfidenzintervall, und hier verwirft der Test 3.700 g
bei $\alpha = 0{,}05$ nicht. Das ist kein Zufall. Das Intervall ist **genau die Menge der
Behauptungen, die der Test nicht verwerfen würde**. Prüf das an einer Reihe von Behauptungen:

In [ ]:
# das 95-%-Konfidenzintervall aus Notebook 02
ci_lower, ci_upper = stats.t.interval(0.95, df=dof, loc=mean_mass, scale=se_mass)
print(f"95-%-Konfidenzintervall: {ci_lower:.0f} g bis {ci_upper:.0f} g\n")

for claim in [3600, 3625, 3650, 3700, 3800, 3825, 3850]:
    p = stats.ttest_1samp(mass, popmean=claim).pvalue
    inside = ci_lower <= claim <= ci_upper
    print(f"Behauptung {claim} g: p = {p:.3f}, "
          f"{'nicht verwerfen' if p >= alpha else 'verwerfen'}, "
          f"{'innerhalb' if inside else 'außerhalb'} des Intervalls")

Jede Behauptung innerhalb des Intervalls hat $p \geq 0{,}05$, und jede außerhalb hat
$p < 0{,}05$. Ein Konfidenzintervall und ein Test sind zwei Blickwinkel auf dieselbe Rechnung:
Das Intervall sagt, *welche Werte plausibel sind*, der Test sagt, *ob ein bestimmter Wert es
ist*.

> **Übung:**
>
> Eine zweite Quelle behauptet, dass die Flossen der Zügelpinguine im Schnitt **200 mm** lang
> sind.
>
> 1. Schreib $H_0$ und $H_1$ auf.
> 2. Berechne die t-Statistik und den zweiseitigen p-Wert **von Hand**, wie in den Abschnitten
>    2 bis 4.
> 3. Überprüfe beide mit `stats.ttest_1samp`.
> 4. Formuliere deine Entscheidung bei $\alpha = 0{,}05$ in einem Satz, den auch jemand ohne
>    Statistikkenntnisse versteht.

Deine Hypothesen hier:

In [ ]:
# dein Code: t-Statistik und p-Wert von Hand

In [ ]:
# dein Code: derselbe Test mit scipy

Deine Schlussfolgerung hier:

## 6. Einseitig oder zweiseitig?

Bisher sagte $H_1$ *anders*, und eine Überraschung in beide Richtungen zählte. Manchmal hat die
Frage nur eine Richtung. Ein Bestimmungsbuch könnte behaupten, dass Eselspinguine (Gentoo) im
Schnitt **mehr als 5.000 g** wiegen; dann ist nur eine schwere Stichprobe ein Beleg dafür, eine
leichte nicht.

Das ist ein **einseitiger Test**. Die gesamten 5 Prozent des Ablehnungsbereichs wandern in einen
Rand, und der p-Wert wird zur Fläche in diesem einen Rand allein. In scipy gibst du die Seite mit
`alternative` an: `"greater"`, `"less"` oder den Standardwert `"two-sided"`.

In [ ]:
# noch einmal der Test für die Zügelpinguine, diesmal nur mit der Frage „schwerer als 3.700 g?“
two_tailed = stats.ttest_1samp(mass, popmean=mu_0)
one_tailed = stats.ttest_1samp(mass, popmean=mu_0, alternative="greater")
print(f"zweiseitiger p-Wert: {two_tailed.pvalue:.3f}")
print(f"einseitiger p-Wert : {one_tailed.pvalue:.3f}")

Der einseitige p-Wert ist genau die Hälfte des zweiseitigen, weil er einen Rand statt zwei
zählt. Dadurch ist ein einseitiger Test leichter zu bestehen, und genau darin liegt seine Gefahr.

> **Achtung.** Wähle die Seite anhand der **Frage**, bevor du die Daten siehst. Sie hinterher zu
> wählen, weil die Stichprobe zufällig schwer ausgefallen ist, ist eine stille Art, deinen
> p-Wert zu halbieren, und so gefundene Ergebnisse halten nicht.

> **Übung:**
>
> Teste die Behauptung über die Eselspinguine anhand der 123 Tiere mit erfasstem Körpergewicht.
>
> 1. Führe den zweiseitigen Test für $\mu = 5{.}000$ g und den einseitigen Test für
>    $\mu > 5{.}000$ g durch.
> 2. Zu welchem Schluss kommt jeder der beiden bei $\alpha = 0{,}05$?
> 3. Welcher beantwortet die Behauptung des Bestimmungsbuchs, und warum ist es *nicht* in
>    Ordnung, zuerst den zweiseitigen Test zu rechnen und zu wechseln, wenn er scheitert?

In [ ]:
# dein Code: zweiseitiger und einseitiger Test für das Körpergewicht der Eselspinguine

Deine Antwort auf Frage 3 hier:

## 7. Von einer Gruppe zu zwei

Die meisten Fragen in einer Analyse vergleichen nicht eine Gruppe mit einer behaupteten Zahl,
sondern **zwei Gruppen miteinander**. Wiegen Adeliepinguine (Adelie) und Zügelpinguine gleich
viel? Die Logik ändert sich überhaupt nicht:

- $H_0$: Die beiden Arten haben dasselbe mittlere Gewicht, der Unterschied ist null.
- Die Teststatistik ist die Differenz der beiden Stichprobenmittelwerte, geteilt durch *ihren*
  Standardfehler.
- Der p-Wert ist wieder eine Fläche unter einer t-Verteilung.

Der Test ist der **Zweistichproben-t-Test**, `stats.ttest_ind`. Die Option `equal_var=False`
macht daraus den **Welch-t-Test**, der nicht annimmt, dass beide Gruppen gleich stark streuen;
er ist die sichere Standardwahl, und Notebook 04 erklärt genauer, warum.

In [ ]:
adelie_mass = df_penguins.loc[df_penguins["species"] == "Adelie", "body_mass_g"].dropna()
gentoo_mass = df_penguins.loc[df_penguins["species"] == "Gentoo", "body_mass_g"].dropna()

for name, other in [("Chinstrap", mass), ("Gentoo", gentoo_mass)]:
    result = stats.ttest_ind(adelie_mass, other, equal_var=False)
    print(f"Adelie ({adelie_mass.mean():.0f} g) vs {name} ({other.mean():.0f} g): "
          f"p = {result.pvalue:.3g}")

Adelie- und Zügelpinguine unterscheiden sich um etwa 30 g, und $p = 0{,}59$: kein Hinweis auf
irgendeinen Unterschied. Adelie- und Eselspinguine unterscheiden sich um deutlich mehr als ein
Kilogramm, und der p-Wert ist so klein, dass er in Zehnerpotenzen geschrieben wird. Beide
Tatsachen hast du im Histogramm am Ende von Notebook 01 gesehen; jetzt kommt zu jeder eine Zahl,
die sagt, wie weit der Zufall sie erklären könnte.

Das ist der Schritt, den Notebook 04 macht: von Pinguinen zu jedem beliebigen Spaltenpaar in
einer explorativen Analyse.

## 8. Die Fehlalarmrate, per Simulation

Abschnitt 4 hat behauptet, dass ein Test bei $\alpha = 0{,}05$ eine **wahre** Nullhypothese in
5 Prozent der Fälle verwirft. Wie in Notebook 02 kannst du das mit einer ausgedachten
Grundgesamtheit überprüfen, denn dort legst du fest, was wahr ist.

> **Übung:**
>
> Erzeuge eine ausgedachte Grundgesamtheit, in der die Nullhypothese **stimmt**: Zügelpinguine
> wiegen wirklich im Schnitt 3.700 g, mit einer Standardabweichung von 384 g.
>
> 1. Ziehe daraus 1.000 Stichproben von je 68 Pinguinen.
> 2. Teste jede Stichprobe mit `stats.ttest_1samp` gegen $\mu_0 = 3{.}700$ g.
> 3. Zähle, wie viele p-Werte unter 0,05 liegen.
>
> Stell dir jetzt eine Analystin vor, die 20 verschiedene Tests auf Daten rechnet, in denen
> nichts passiert. Wie viele „signifikante“ Ergebnisse sollte sie erwarten?

<details><summary>
Hier klicken für einen Tipp ...
</summary>

`rng.normal(loc=3700, scale=384, size=(1000, 68))` zieht alle 1.000 Stichproben auf einmal, eine
pro Zeile. `stats.ttest_1samp(samples, popmean=3700, axis=1)` testet dann jede Zeile in einem
Aufruf, und `.pvalue` enthält alle 1.000 p-Werte.
</details>

In [ ]:
rng = np.random.default_rng(7)

# dein Code: 1.000 Stichproben, in denen H0 stimmt, je ein Test und der Anteil mit p < 0.05

## Zusammenfassung

In diesem Notebook hast du einen Hypothesentest von Grund auf durchgeführt:

1. **Hypothesen.** $H_0$, die langweilige Erklärung, und $H_1$, das, worauf du stattdessen
   schließen würdest, beide über die Grundgesamtheit.
2. **Signifikanzniveau.** $\alpha = 0{,}05$, vorab festgelegt: die Fehlalarmrate, die du
   akzeptierst.
3. **Teststatistik.** Wie viele Standardfehler die Stichprobe von $H_0$ entfernt liegt.
4. **p-Wert.** Die Fläche jenseits davon unter der t-Verteilung: wie überraschend die Daten
   sind, wenn $H_0$ stimmt.
5. **Entscheidung.** Verwerfen, wenn $p < \alpha$; sonst nicht verwerfen, was nicht dasselbe ist,
   wie $H_0$ zu beweisen.

Jeder Test, dem du begegnest, folgt diesen fünf Schritten. Nur die Teststatistik und ihre Kurve
ändern sich, und um beides kümmert sich scipy. Außerdem hast du gesehen, dass ein Test und ein
Konfidenzintervall immer übereinstimmen, dass die Seite gewählt werden muss, bevor man die Daten
sieht, und dass dieselbe Logik zwei Gruppen vergleicht.

Als Nächstes setzt `04_hypothesis_testing_for_eda.ipynb` das dort ein, wo du es am meisten
brauchst: bei den Mustern, die eine explorative Analyse zutage fördert.

## Quellen & weiterführende Literatur

- [**scipy.stats.ttest_1samp**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_1samp.html):
  Der Einstichproben-t-Test, inklusive des Arguments `alternative` aus Abschnitt 6.
- [**scipy.stats.ttest_ind**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html):
  Der Zweistichproben-t-Test aus Abschnitt 7; `equal_var=False` liefert die Welch-Variante.
- [**StatQuest: Hypothesis Testing and The Null Hypothesis**](https://www.youtube.com/watch?v=0oc49DyA3hU):
  Abschnitt 1 in fünfzehn Minuten.
- [**StatQuest: p-values, what they are and how to interpret them**](https://www.youtube.com/watch?v=vemZtEM63GY):
  Was ein p-Wert aussagt und was nicht.
- [**ModernDive: Hypothesis Testing**](https://moderndive.com/v2/hypothesis-testing.html):
  Dieselbe Logik, mit Simulation statt mit Formeln aufgebaut.